# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 4 — CTR / Engagement Opportunity Scoring (position-adjusted).**

I picked Lane 4 because the starter data shows a large, *measurable* pool of pages that already rank well but under-capture clicks relative to their peers. That is the kind of problem a ranked, leakage-safe scoring queue is built for: the demand is there, the position is there, but the click yield is not. Unlike a generic "predict decline" framing, a position-adjusted CTR gap gives a reviewer an immediate, defensible action (fix title/meta/snippet for page X) and a natural quality filter (require real impressions, so we are not chasing noise). The warehouse release later lets me compute *expected* CTR per position tier from millions of daily rows instead of this 30k slice, so the gap baseline gets much stronger — but the shape of the problem is already visible in the starter data.

I can confirm or switch lanes until the end of Week 4; this is a provisional choice backed by numbers below, not a commitment.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

import os
csv_path = "data/raw/content_refresh_anonymized.csv"
if not os.path.exists(csv_path):
    csv_path = os.path.join("..", "..", csv_path)
df = pd.read_csv(csv_path)
print("rows, cols:", df.shape)
print("distinct clients:", df['client_id'].nunique())

# Recall the lane guide numbers so the choice is anchored in real data.
measurable = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0)]
print("\nPages with >=500 impressions and a real position (>=0):", len(measurable))
print(measurable['position_tier'].value_counts())

# Lane 4 requires comparing pages only WITHIN their position tier (a page at
# position 1 always beats a page at position 9 on raw CTR).
tier_median = measurable.groupby('position_tier')['ctr'].median()
print("\nMedian CTR by position tier:")
print(tier_median.round(3))

rows, cols: (30000, 44)
distinct clients: 32

Pages with >=500 impressions and a real position (>=0): 16726
position_tier
page_1      7064
striking    4485
page_3_5    4330
top_3        458
deep         389
Name: count, dtype: int64

Median CTR by position tier:
position_tier
deep        0.00
page_1      0.24
page_3_5    0.09
striking    0.17
top_3       0.20
Name: ctr, dtype: float64


## 2. The question: decision, action, cost of a wrong call

**Decision improved:** which visible pages should an editor or SEO reviewer look at *first* to improve click-through and on-page engagement — given limited review capacity (say, the top 20–50 pages per cycle).

**Who acts, and what they do:** a content/SEO reviewer opens a ranked queue and rewrites titles, meta descriptions, or snippet structure (and, for engagement candidates, on-page layout) for the pages at the top of the list. Each row in the output is *one page* (the grain).

**Cost of a wrong call:**
- *False positive (we flag a page that was actually fine):* wasted reviewer hours on a page whose CTR was low for a legitimate reason (e.g. a navigational query where users don't need to click). Low-to-moderate cost, but it scales — if the queue is noisy, reviewers stop trusting it.
- *False negative (we miss a page bleeding clicks):* a high-demand page keeps under-capturing clicks for months. This is the more expensive error: real, measurable traffic and potential conversions left on the table, repeated every cycle.

So the metric that matches the decision is **precision@K** (of the top-K pages I hand a reviewer, how many are genuinely below their tier's expected CTR) plus a **recall** view so we don't silently ignore too many real opportunities. This is decision-support, not automation: a human still reviews and decides.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quantify the opportunity pool that motivates the cost-of-missing framing.
page1 = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 10)]
print("Page-1 visible pages (>=500 imp, position 1-10):", len(page1))
low_ctr = page1[page1['ctr'] < 0.5]
print("of those, with CTR < 0.5% (raw, unadjusted):", len(low_ctr),
      f"({len(low_ctr)/len(page1):.1%})")

# Position-adjusted: how many sit well below their OWN tier's median CTR?
# This is the honest Lane-4 filter (compare like-with-like).
m = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0)].copy()
tier_med = m.groupby('position_tier')['ctr'].transform('median')
m['below_70pct_tier'] = m['ctr'] < 0.7 * tier_med
print("\nPages >=500 imp, real position, below 70% of their tier median CTR:",
      int(m['below_70pct_tier'].sum()), f"({m['below_70pct_tier'].mean():.1%})")

Page-1 visible pages (>=500 imp, position 1-10): 7564
of those, with CTR < 0.5% (raw, unadjusted): 5969 (78.9%)

Pages >=500 imp, real position, below 70% of their tier median CTR: 6058 (36.2%)


## 3. Quick look at the data (2-3 real numbers)

Backing numbers, all from `data/raw/content_refresh_anonymized.csv` (30,000 pages, 32 clients):

1. **~16,726 pages (55.8%)** carry ≥500 impressions *and* a real position (`avg_position > 0`) — a big, demand-bearing pool where a CTR comparison is even meaningful (CTR on a 3-impression page is noise).
2. **~35% of those visible pages sit below 70% of their own position-tier median CTR.** That is a large, position-adjusted opportunity pool — not an artifact of comparing a position-9 page to a position-1 page.
3. **7,564 pages rank on page 1 (position 1–10) with ≥500 impressions, and 78.9% of them have raw CTR < 0.5%.** Page-1 real estate is the highest-value place to lose clicks, so this is where review capacity should go first.

These numbers say Lane 4 is worth the next weeks: the signal is real, the action is concrete, and a minimum-volume filter already keeps most noise out.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Print the three backing numbers cleanly so the notebook stands alone.
n = len(df)
measurable = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0)]
m = measurable.copy()
tier_med = m.groupby('position_tier')['ctr'].transform('median')
m['below_70pct_tier'] = m['ctr'] < 0.7 * tier_med
page1 = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 10)]
low_ctr_page1 = page1[page1['ctr'] < 0.5]

print(f"1) Visible pages (>=500 imp, real position): {len(measurable)} ({len(measurable)/n:.1%} of all)")
print(f"2) Of those, below 70% of tier-median CTR: {int(m['below_70pct_tier'].sum())} ({m['below_70pct_tier'].mean():.1%})")
print(f"3) Page-1 visible pages with CTR<0.5%: {len(low_ctr_page1)} of {len(page1)} ({len(low_ctr_page1)/len(page1):.1%})")

1) Visible pages (>=500 imp, real position): 16726 (55.8% of all)
2) Of those, below 70% of tier-median CTR: 6058 (36.2%)
3) Page-1 visible pages with CTR<0.5%: 5969 of 7564 (78.9%)


## 4. Careful words: what I can and can't claim

**What this work can say (careful language):**
- *Observed:* in this data, a measurable share of visible pages under-capture clicks relative to same-tier peers.
- *Directional / decision-support:* a ranked list can help a reviewer spend limited time on the most promising pages first; the top of the list is where review effort is most likely to pay off.
- I will report precision@K against a transparent baseline and a human by-hand review of the top 20.

**What this work can NEVER claim:**
- *Causation:* I cannot claim that rewriting a title *caused* a CTR lift — proving that needs an experiment I don't have. My output is a *review queue*, not a guarantee.
- *"Predicting Google":* I am not discovering a search-ranking factor or an algorithm rule.
- *Leakage:* `trend_direction` / `trend_pct` are the label source and are **never** features; I will not compare CTR across position tiers without adjusting for position; I will use a client-holdout split so pages from the same client don't leak across train/test.
- *Sparse-signal overreach:* I will not build a binary classifier on AI-session data alone (too sparse) — if I touch AI referral it stays EDA only.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Guardrail check: confirm the leakage columns are NOT in our feature thinking.
leakage_cols = ['trend_direction', 'trend_pct', 'is_declining_label']
present = [c for c in leakage_cols if c in df.columns]
print("Leakage-source columns present in raw data (must be excluded as features):", present)
# And confirm we only ever compare CTR within position tiers, not across them.
print("Position tiers present:", sorted(df['position_tier'].dropna().unique().tolist()))

Leakage-source columns present in raw data (must be excluded as features): ['trend_direction', 'trend_pct']
Position tiers present: ['deep', 'page_1', 'page_3_5', 'striking', 'top_3']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.